# **MODELOS DE CLASSIFICAÇÃO: RANDOM FOREST E BOOSTING**
### *Estudo de Caso G: Reinternação Hospitalar em 30 dias*
---

**Objetivo:** estimar, no momento da alta, a probabilidade de cada paciente ter uma internação não programada no hospital nos 30 dias seguintes (`reinternacao_30d = 1`), para priorizar quem entra no programa de transição de cuidado (ligações da enfermagem e consulta precoce), que tem capacidade limitada.

### **Bibliotecas**

In [ ]:
# Instalação de bibliotecas não disponíveis na instalação padrão do Python no Google Colab
# %pip install shap xgboost lightgbm

In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform, loguniform
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import get_scorer
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.inspection import permutation_importance
import shap
import time

In [ ]:
os.environ["OMP_NUM_THREADS"] = "1"
np.random.seed(123)

### **Leitura da base de dados**

In [ ]:
arquivo = "Estudo_Caso_5_Base_G_Dados.txt"

if not os.path.exists(arquivo):
    from google.colab import files
    files.upload()

dados = pd.read_table(arquivo,
                      sep="\t",
                      decimal=".",
                      header=0)

### **Visualização da base de dados**

In [ ]:
dados.head(10)

### **Dimensões da base de dados**

In [ ]:
dados.shape

### **Tipos das colunas da base de dados**

In [ ]:
dados.dtypes

### **Análise exploratória inicial**

*Duplicidades*

In [ ]:
print(f"Linhas duplicadas: {dados.duplicated().sum()}")
print(f"Internações repetidas: {dados['id_internacao'].duplicated().sum()}")

*Análise de preenchimento*

In [ ]:
dados.isna().sum()

*Taxa de reinternação entre registros com e sem preenchimento*

In [ ]:
for var in ['mora_sozinho', 'hemoglobina_alta', 'sodio_alta']:
    resumo = dados.groupby(dados[var].isna())['reinternacao_30d'].agg(['size', 'mean'])
    resumo.index = resumo.index.map({False: 'preenchido', True: 'ausente'})
    resumo.columns = ['qtd', 'taxa_reinternacao']
    print(var)
    print(resumo.round(3), '\n')

*Perfil das internações sem exames na alta*

In [ ]:
exames_faltantes = dados['hemoglobina_alta'].isna() | dados['sodio_alta'].isna()

print(pd.crosstab(dados['tipo_admissao'] + ' / cirurgia: ' + dados['realizou_cirurgia'],
                  exames_faltantes.map({False: 'com exames', True: 'sem exames'})))
print()
print(dados.groupby(exames_faltantes.map({False: 'com exames', True: 'sem exames'}))['tempo_internacao_dias'].describe().round(1))

*Tratamento de valores ausentes e criação de variáveis*

Os ausentes não são aleatórios: quem não informou se mora sozinho reinterna mais do que quem informou "nao", e os exames faltam quase só em internações eletivas curtas (cirurgias de 1 a 2 dias), com taxa de reinternação bem menor. Por isso as linhas não são removidas: `mora_sozinho` ganha a categoria `nao_informado`, a falta de exame vira a indicadora `exame_ausente` e os valores de hemoglobina e sódio são imputados pela mediana do treino dentro do pré-processamento. O dia da alta é resumido em `alta_fim_semana`, já que as altas de sábado e domingo concentram as maiores taxas.

In [ ]:
def preparar_base(base):
    base = base.copy()
    base['mora_sozinho'] = base['mora_sozinho'].fillna('nao_informado')
    base['exame_ausente'] = (base['hemoglobina_alta'].isna() | base['sodio_alta'].isna()).astype(int)
    base['alta_fim_semana'] = np.where(base['dia_semana_alta'].isin(['sabado', 'domingo']), 'sim', 'nao')
    return base

dados = preparar_base(dados)
dados[['mora_sozinho', 'exame_ausente', 'alta_fim_semana']].apply(lambda s: s.value_counts().to_dict())

*Estatísticas descritivas das variáveis quantitativas*

In [ ]:
dados.describe().T.round(2)

*Análise da distribuição da variável resposta*

In [ ]:
frequencias = dados['reinternacao_30d'].value_counts().sort_index()
proporcoes = dados['reinternacao_30d'].value_counts(normalize=True).sort_index() * 100

plt.figure(figsize=(10, 3.5))
barras = plt.barh(frequencias.index.astype(str), frequencias.values,
                  color=['#C3FA75', '#C44E52'], edgecolor='#084734', height=0.5)

for barra, frequencia, proporcao in zip(barras, frequencias.values, proporcoes.values):
    plt.text(frequencia + frequencias.max() * 0.02,
             barra.get_y() + barra.get_height() / 2,
             f"{frequencia} ({proporcao:.1f}%)", va='center')

plt.title('Distribuição: reinternacao_30d')
plt.xlabel('Frequência')
plt.ylabel('reinternacao_30d')
plt.xlim(0, frequencias.max() * 1.25)
plt.show()

print(pd.DataFrame({'Frequência': frequencias,
                    'Proporção (%)': proporcoes.round(2)}))

*Recodificação da variável resposta, se necessário*

In [ ]:
# Já vem codificada como 0 e 1, com 1 = reinternação em 30 dias
sorted(dados['reinternacao_30d'].unique())

### **Definição de variáveis explicativas e variável resposta**

*Lista de nomes das variáveis explicativas, separando em quantitativas e qualitativas*

In [ ]:
lista_X_quanti = ['idade',
                  'distancia_hospital_km',
                  'indice_charlson',
                  'qtd_internacoes_12m',
                  'qtd_atendimentos_ps_6m',
                  'tempo_internacao_dias',
                  'hemoglobina_alta',
                  'sodio_alta',
                  'qtd_medicamentos_alta',
                  'exame_ausente'
                  ]

lista_X_quali = ['sexo',
                 'categoria_plano',
                 'mora_sozinho',
                 'tipo_admissao',
                 'clinica_responsavel',
                 'passou_uti',
                 'realizou_cirurgia',
                 'alta_fim_semana',
                 'destino_alta',
                 'retorno_agendado'
                 ]

*Criação de objetos para explicativas (X) e resposta (y)*

In [ ]:
y = dados['reinternacao_30d']
X = dados[lista_X_quanti + lista_X_quali]

### **Análise bivariada: variável resposta vs. variáveis explicativas**

*Gráficos de boxplot das variáveis quantitativas, por categoria da variável resposta*

In [ ]:
n = len(lista_X_quanti)

ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).flatten()

for ax, var in zip(axes, lista_X_quanti):
    sns.boxplot(x=y.name, y=var, data=dados, ax=ax, hue=y.name, legend=False,
                palette=['#C3FA75', '#C44E52'], showfliers=False, width=0.55, dodge=False,
                medianprops=dict(color='#084734'))
    ax.set_title(var)
    ax.set_xlabel('')

for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

*Média das variáveis quantitativas por categoria da variável resposta*

In [ ]:
dados.groupby('reinternacao_30d')[lista_X_quanti].mean().T.round(2)

*Gráficos de barras com a taxa do evento, para as variáveis qualitativas*

In [ ]:
n = len(lista_X_quali)

ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).flatten()

for ax, var in zip(axes, lista_X_quali):
    taxas = dados.groupby(var)[y.name].mean() * 100
    barras = ax.barh(taxas.index.astype(str), taxas.values, color='#C3FA75',
                     edgecolor='#084734')
    for barra, valor in zip(barras, taxas.values):
        ax.text(valor, barra.get_y() + barra.get_height() / 2,
                f' {valor:.1f}%', ha='left', va='center')
    ax.set_title(var)
    ax.set_xlabel('Taxa de reinternação (%)')
    ax.set_xlim(0, taxas.max() * 1.2)

for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

### **Divisão de treino e teste externo**

In [ ]:
X_treino, X_teste_externo, y_treino, y_teste_externo = train_test_split(X, y, test_size=0.2, stratify=y, random_state=123)
print(X_treino.shape, X_teste_externo.shape)
print(f"Taxa de reinternação: treino {y_treino.mean():.2%} | teste externo {y_teste_externo.mean():.2%}")

### **Pré-processamento**

In [ ]:
# Mediana calculada só no treino; árvores não dependem da escala, então as quantitativas não são padronizadas
preprocessador = ColumnTransformer(transformers=[
    ("quanti", SimpleImputer(strategy="median"), lista_X_quanti),
    ("quali", OneHotEncoder(sparse_output=False, drop="first", handle_unknown='ignore'), lista_X_quali)
], verbose_feature_names_out=False)
preprocessador.set_output(transform="pandas")

X_treino_tratada = preprocessador.fit_transform(X_treino)
X_teste_externo_tratada = preprocessador.transform(X_teste_externo)
X_completa_tratada = preprocessador.transform(X)

X_treino_tratada.shape

### **Construção do modelo (com validação cruzada e otimização de hiperparâmetros)**

*Definição da(s) técnica(s) de modelagem*

In [ ]:
# Com 8,6% de reinternações, os pesos balanceados entram dentro do fit para valer também nos folds e na calibração
class ModeloPonderado(ClassifierMixin, BaseEstimator):
    def __init__(self, modelo):
        self.modelo = modelo
    def fit(self, X, y):
        self.modelo_ = clone(self.modelo).fit(X, y, sample_weight=compute_sample_weight(class_weight='balanced', y=y))
        self.classes_ = self.modelo_.classes_
        return self
    def predict_proba(self, X):
        return self.modelo_.predict_proba(X)
    def predict(self, X):
        return self.modelo_.predict(X)

tecnicas = {

    'Arvore de Decisao': (DecisionTreeClassifier(random_state=123), {
        'criterion': ['gini', 'entropy'],
        'min_impurity_decrease': uniform(0, 0.05),
        'min_samples_split': randint(100, 201),
        'min_samples_leaf': randint(50, 101),
        'max_depth': randint(2, 11)
    }),

    'Random Forest': (RandomForestClassifier(random_state=123), {
        'n_estimators': randint(100, 501),
        'criterion': ['gini', 'entropy'],
        'min_impurity_decrease': uniform(0, 0.05),
        'min_samples_split': randint(100, 201),
        'min_samples_leaf': randint(50, 101),
        'max_depth': randint(2, 11),
        'max_samples': uniform(0.6, 0.4),
        'max_features': ['sqrt', 'log2', None]
    }),

    'AdaBoost': (AdaBoostClassifier(random_state=123), {
        'n_estimators': randint(100, 201),
        'learning_rate': loguniform(0.03, 1)
    }),

    'Gradient Boosting': (GradientBoostingClassifier(random_state=123), {
        'n_estimators': randint(100, 501),
        'learning_rate': loguniform(0.03, 0.3),
        'min_samples_split': randint(100, 201),
        'min_samples_leaf': randint(50, 101),
        'subsample': uniform(0.6, 0.4),
        'max_features': ['sqrt', 'log2', None],
        'max_depth': randint(2, 11)
    }),

    'XGBoost': (XGBClassifier(random_state=123, verbosity=0, n_jobs=1), {
        'n_estimators': randint(100, 501),
        'learning_rate': loguniform(0.03, 0.3),
        'colsample_bytree': uniform(0.5, 0.5),
        'subsample': uniform(0.6, 0.4),
        'gamma': uniform(0, 5),
        'reg_alpha': loguniform(1e-5, 100),
        'reg_lambda': loguniform(1e-5, 100),
        'max_depth': randint(2, 11)
    }),

    'Hist. Gradient Boosting': (HistGradientBoostingClassifier(random_state=123), {
        'max_iter': randint(100, 501),
        'learning_rate': loguniform(0.03, 0.3),
        'min_samples_leaf': randint(50, 101),
        'max_leaf_nodes': randint(15, 64),
        'max_bins': randint(64, 256),
        'l2_regularization': loguniform(1e-5, 100)
    }),

    'LightGBM': (LGBMClassifier(random_state=123, verbose=-1, subsample_freq=1, importance_type='gain', n_jobs=1), {
        'n_estimators': randint(100, 501),
        'learning_rate': loguniform(0.03, 0.3),
        'min_child_samples': randint(50, 101),
        'num_leaves': randint(15, 64),
        'max_bin': randint(64, 256),
        'subsample': uniform(0.6, 0.4),
        'reg_alpha': loguniform(1e-5, 100),
        'reg_lambda': loguniform(1e-5, 100)
    })

}

*Estratégia de validação cruzada (k-fold estratificado), quantidade de iterações de busca e métricas de avaliação*

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
qtd_iteracoes = 50
lista_metricas = ['roc_auc',
                  'balanced_accuracy']
scorers = {metrica: get_scorer(metrica) for metrica in lista_metricas}
sinais = {metrica: -1 if metrica.startswith('neg_') else 1 for metrica in lista_metricas}

*Construção dos modelos*

In [ ]:
resultados = []
contador_modelos = 0
pesos_treino = compute_sample_weight(class_weight='balanced', y=y_treino)

for nome_tecnica, (classe_tecnica, hiperparametros) in tecnicas.items():

    inicio = time.time()

    busca = RandomizedSearchCV(
        estimator=ModeloPonderado(classe_tecnica),
        param_distributions={'modelo__' + nome: valores for nome, valores in hiperparametros.items()},
        n_iter=1 if not hiperparametros else qtd_iteracoes,
        scoring=lista_metricas,
        refit=False,
        cv=kf,
        return_train_score=True,
        random_state=123,
        n_jobs=-1
    )

    busca.fit(X_treino_tratada, y_treino)

    resultados_aux = pd.DataFrame({
        "num_modelo": [f"Modelo {contador_modelos + i}" for i in range(len(busca.cv_results_["params"]))],
        "nome_tecnica": nome_tecnica,
        "classe_tecnica": type(classe_tecnica).__name__,
        "hiperparametros": [{nome.replace("modelo__", ""): valor for nome, valor in params.items()}
                            for params in busca.cv_results_["params"]]
    })

    metricas_teste_externo = {metrica: [] for metrica in lista_metricas}
    for params in resultados_aux["hiperparametros"]:
        estimador = clone(classe_tecnica).set_params(**params)
        estimador.fit(X_treino_tratada, y_treino, sample_weight=pesos_treino)
        for metrica in lista_metricas:
            metricas_teste_externo[metrica].append(sinais[metrica] * scorers[metrica](estimador, X_teste_externo_tratada, y_teste_externo))

    for metrica in lista_metricas:
        nome_metrica = metrica.replace('neg_', '')
        resultados_aux[f"{nome_metrica}_media_treino"] = sinais[metrica] * busca.cv_results_[f"mean_train_{metrica}"]
        resultados_aux[f"{nome_metrica}_dp_treino"] = busca.cv_results_[f"std_train_{metrica}"]
        resultados_aux[f"{nome_metrica}_media_teste_interno"] = sinais[metrica] * busca.cv_results_[f"mean_test_{metrica}"]
        resultados_aux[f"{nome_metrica}_dp_teste_interno"] = busca.cv_results_[f"std_test_{metrica}"]
        resultados_aux[f"{nome_metrica}_teste_externo"] = metricas_teste_externo[metrica]
        resultados_aux[f"{nome_metrica}_var_teste_interno"] = abs(resultados_aux[f"{nome_metrica}_media_teste_interno"] - resultados_aux[f"{nome_metrica}_media_treino"])
        resultados_aux[f"{nome_metrica}_var_perc_teste_interno"] = resultados_aux[f"{nome_metrica}_var_teste_interno"] / abs(resultados_aux[f"{nome_metrica}_media_treino"])
        resultados_aux[f"{nome_metrica}_var_teste_externo"] = abs(resultados_aux[f"{nome_metrica}_teste_externo"] - resultados_aux[f"{nome_metrica}_media_treino"])
        resultados_aux[f"{nome_metrica}_var_perc_teste_externo"] = resultados_aux[f"{nome_metrica}_var_teste_externo"] / abs(resultados_aux[f"{nome_metrica}_media_treino"])

    resultados.append(resultados_aux)
    contador_modelos += len(resultados_aux)

    fim = time.time()
    print(f"({math.ceil(contador_modelos/qtd_iteracoes)} de {len(tecnicas)}) Processo concluído para a técnica {nome_tecnica} em {fim - inicio:.1f} segundos!")

resultados = pd.concat(resultados, ignore_index=True)

*Resultados completos da execução*

In [ ]:
resultados.head(10)

### **Comparação de modelos**

*Escolha de critérios*

In [ ]:
qtd_modelos = 15
ordenar_por = "roc_auc"

# Diferença máxima aceita entre treino e testes (0.1 = 10%), para descartar modelos sobreajustados
limite_var_perc_metrica_teste_interno = 0.1
limite_var_perc_metrica_teste_externo = 0.1

*Ordenação e filtragem dos melhores modelos*

In [ ]:
nome_metrica_ordenacao = ordenar_por.replace('neg_', '')
crescente = ordenar_por.startswith('neg_')
resultados_filtrados = resultados[(resultados[f'{nome_metrica_ordenacao}_var_perc_teste_interno'] < limite_var_perc_metrica_teste_interno) & (resultados[f'{nome_metrica_ordenacao}_var_perc_teste_externo'] < limite_var_perc_metrica_teste_externo)]
resultados_filt_ord = resultados_filtrados.sort_values(f'{nome_metrica_ordenacao}_media_teste_interno', ascending=crescente).head(qtd_modelos)
print(f"Modelos que passaram no filtro de estabilidade: {len(resultados_filtrados)} de {len(resultados)}")

*Exibição dos melhores modelos, em forma de tabela*

In [ ]:
pd.set_option('display.float_format', '{:.3f}'.format)
resultados_filt_ord[['num_modelo',
                     'nome_tecnica',
                     'hiperparametros',
                     f'{nome_metrica_ordenacao}_media_treino',
                     f'{nome_metrica_ordenacao}_dp_treino',
                     f'{nome_metrica_ordenacao}_media_teste_interno',
                     f'{nome_metrica_ordenacao}_dp_teste_interno',
                     f'{nome_metrica_ordenacao}_var_perc_teste_interno',
                     f'{nome_metrica_ordenacao}_teste_externo',
                     f'{nome_metrica_ordenacao}_var_perc_teste_externo']]

*Melhor modelo de cada técnica (após o filtro)*

In [ ]:
(resultados_filtrados.sort_values(f'{nome_metrica_ordenacao}_media_teste_interno', ascending=crescente)
                     .groupby('nome_tecnica').head(1)
                     [['num_modelo', 'nome_tecnica',
                       f'{nome_metrica_ordenacao}_media_treino',
                       f'{nome_metrica_ordenacao}_media_teste_interno',
                       f'{nome_metrica_ordenacao}_teste_externo',
                       'balanced_accuracy_media_teste_interno']])

*Exibição dos melhores modelos, em forma de gráfico*

In [ ]:
dados_grafico = resultados_filt_ord.iloc[::-1]
y_pos = np.arange(len(dados_grafico))

plt.figure(figsize=(12, 6))

plt.errorbar(dados_grafico[f"{nome_metrica_ordenacao}_media_teste_interno"], y_pos,
             xerr=dados_grafico[f"{nome_metrica_ordenacao}_dp_teste_interno"],
             fmt='none', ecolor='#4860E7', elinewidth=8, alpha=0.25)

plt.scatter(dados_grafico[f"{nome_metrica_ordenacao}_media_treino"], y_pos,
            color='black', marker='|', s=150, linewidths=2, label='Treino (média)', zorder=4)
plt.scatter(dados_grafico[f"{nome_metrica_ordenacao}_media_teste_interno"], y_pos,
            color="#4860E7", s=100, label='Teste interno (média ± dp)', zorder=3)
plt.scatter(dados_grafico[f"{nome_metrica_ordenacao}_teste_externo"], y_pos,
            color="#DD7D45", edgecolor='white', s=60, label='Teste externo', zorder=3)

plt.axvline(dados_grafico[f"{nome_metrica_ordenacao}_media_teste_interno"].iloc[-1],
            color='black', linestyle='--', linewidth=1, alpha=0.5)

plt.yticks(y_pos, dados_grafico["num_modelo"] + " (" + dados_grafico["nome_tecnica"] + ")")
plt.xlabel(nome_metrica_ordenacao)
plt.title("Comparativo dos melhores modelos: treino, teste interno e teste externo")
plt.legend(loc='lower right', labelspacing=0.8)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### **Análise diagnóstica**

*Escolha do modelo*

In [ ]:
# Primeiro do ranking filtrado; dá para trocar por outro número da tabela acima (ex.: um modelo mais simples com AUC parecida)
num_modelo_escolhido = int(resultados_filt_ord.iloc[0]["num_modelo"].split()[-1])
linha_modelo_escolhido = resultados.loc[resultados["num_modelo"] == 'Modelo ' + str(num_modelo_escolhido)].squeeze()
print(f"Modelo {num_modelo_escolhido} ({linha_modelo_escolhido['nome_tecnica']})")

*Hiperparâmetros principais do modelo escolhido*

In [ ]:
linha_modelo_escolhido["hiperparametros"]

*Medidas de qualidade de ajuste*

In [ ]:
tabela_desempenho = {}
for metrica in lista_metricas:
    nome_metrica = metrica.replace('neg_', '')
    tabela_desempenho[nome_metrica] = {
        'Treino (média)': linha_modelo_escolhido[f"{nome_metrica}_media_treino"],
        'Treino (dp)': linha_modelo_escolhido[f"{nome_metrica}_dp_treino"],
        'Teste interno (média)': linha_modelo_escolhido[f"{nome_metrica}_media_teste_interno"],
        'Teste interno (dp)': linha_modelo_escolhido[f"{nome_metrica}_dp_teste_interno"],
        'Teste externo': linha_modelo_escolhido[f"{nome_metrica}_teste_externo"]
    }
tabela_desempenho = pd.DataFrame(tabela_desempenho).T

print(f"-------------------------- Medidas de qualidade de ajuste: MODELO {num_modelo_escolhido} --------------------------")
tabela_desempenho.round(3)

*Reajuste do modelo escolhido no conjunto de treino completo*

In [ ]:
modelo_escolhido = clone(tecnicas[linha_modelo_escolhido["nome_tecnica"]][0]).set_params(**linha_modelo_escolhido["hiperparametros"])
modelo_escolhido.fit(X_treino_tratada, y_treino, sample_weight=pesos_treino)

*Gráfico preliminar da árvore (aplicável apenas quando o modelo escolhido é de árvore de decisão)*

In [ ]:
if isinstance(modelo_escolhido, DecisionTreeClassifier):

    plt.figure(figsize=(30, 12))
    plot_tree(modelo_escolhido,
              feature_names=X_treino_tratada.columns,
              class_names=modelo_escolhido.classes_.astype(str),
              filled=True,
              proportion=True,
              fontsize=10,
              rounded=True,
              precision=2)
    plt.title('Árvore do modelo escolhido (no conjunto de treino)')
    plt.show()

*Probabilidades preditas no conjunto de teste externo*

In [ ]:
p_hat = modelo_escolhido.predict_proba(X_teste_externo_tratada)[:, 1]
pd.Series(p_hat).describe()

*Curva de calibração e score de Brier, antes da calibração*

In [ ]:
def curva_calibracao(y_binaria, p, faixas=10):
    limites = np.linspace(0, 1, faixas + 1)
    indice = np.clip(np.digitize(p, limites) - 1, 0, faixas - 1)
    tabela = pd.DataFrame({'faixa': indice, 'predita': p,
                           'observada': y_binaria})
    return tabela.groupby('faixa')[['predita', 'observada']].mean()

pontos = curva_calibracao(np.asarray(y_teste_externo), p_hat)
brier = np.mean((p_hat - np.asarray(y_teste_externo)) ** 2)
print(f'Score de Brier: {brier:.3f}')

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.plot(pontos['predita'], pontos['observada'], '-o', color='#084734')
plt.xlabel('Probabilidade média predita')
plt.ylabel('Taxa observada do evento')
plt.title(f'Score de Brier: {brier:.3f}')
plt.show()

*Calibração das probabilidades*

In [ ]:
# A ponderação infla as probabilidades; a calibração isotônica traz de volta para a escala da taxa real
modelo_calibrado = CalibratedClassifierCV(ModeloPonderado(modelo_escolhido),
                                          method='isotonic',
                                          cv=5)
modelo_calibrado.fit(X_treino_tratada, y_treino)
p_hat_calibrada = modelo_calibrado.predict_proba(X_teste_externo_tratada)[:, 1]

*Curva de calibração e score de Brier, depois da calibração*

In [ ]:
pontos = curva_calibracao(np.asarray(y_teste_externo), p_hat_calibrada)
brier_calibrado = np.mean((p_hat_calibrada - np.asarray(y_teste_externo)) ** 2)
print(f'Score de Brier: {brier_calibrado:.3f}')

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.plot(pontos['predita'], pontos['observada'], '-o', color='#084734')
plt.xlabel('Probabilidade média predita')
plt.ylabel('Taxa observada do evento')
plt.title(f'Score de Brier: {brier_calibrado:.3f}')
plt.show()

*Distribuição das probabilidades preditas, antes e depois da calibração*

In [ ]:
taxa_observada = np.asarray(y_teste_externo).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (p, titulo) in zip(axes, ((p_hat, 'Antes da calibração'),
                                  (p_hat_calibrada, 'Depois da calibração'))):
    ax.hist(p, bins=np.linspace(0, 1, 51), color='#C3FA75', edgecolor='#084734')
    ax.axvline(taxa_observada, color='#C44E52', linestyle='--')
    ax.set_title(f'{titulo}: média de {p.mean():.1%}')
    ax.set_xlabel('Probabilidade predita')
    ax.set_xlim(0, 1)

axes[0].set_ylabel('Quantidade de observações')
plt.tight_layout()
plt.show()

print(f'Média das probabilidades preditas, antes da calibração: {p_hat.mean():.1%}')
print(f'Média das probabilidades preditas, depois da calibração: {p_hat_calibrada.mean():.1%}')
print(f'Taxa observada do evento (linha tracejada): {taxa_observada:.1%}')

*Curva ROC, estatística KS e AUC, antes e depois da calibração*

In [ ]:
def curva_ks(y, p):
    acumulado_nao_evento, acumulado_evento, _ = roc_curve(y, p)
    percentual = (acumulado_evento * (y == 1).sum()
                  + acumulado_nao_evento * (y == 0).sum()) / len(y)
    return percentual, acumulado_evento, acumulado_nao_evento

y_binaria = np.asarray(y_teste_externo)
_, evento, nao_evento = curva_ks(y_binaria, p_hat)
percentual, evento_cal, nao_evento_cal = curva_ks(y_binaria, p_hat_calibrada)

discriminancia = pd.DataFrame({
    'Antes da calibração': {'AUC': roc_auc_score(y_binaria, p_hat),
                            'KS': np.max(evento - nao_evento)},
    'Depois da calibração': {'AUC': roc_auc_score(y_binaria, p_hat_calibrada),
                             'KS': np.max(evento_cal - nao_evento_cal)}})
print(discriminancia.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot([0, 1], [0, 1], '--', color='gray')
for p, rotulo, cor in ((p_hat, 'Antes da calibração', '#084734'),
                       (p_hat_calibrada, 'Depois da calibração', '#C44E52')):
    fpr, tpr, _ = roc_curve(y_binaria, p)
    axes[0].plot(fpr, tpr, color=cor,
                 label=f'{rotulo} (AUC = {roc_auc_score(y_binaria, p):.3f})')
axes[0].set_title('Curva ROC')
axes[0].set_xlabel('1 - especificidade')
axes[0].set_ylabel('Sensibilidade')
axes[0].legend()

maior_distancia = np.argmax(evento_cal - nao_evento_cal)
axes[1].plot(percentual, evento_cal, color='#C44E52', label='Com o evento')
axes[1].plot(percentual, nao_evento_cal, color='#084734', label='Sem o evento')
axes[1].vlines(percentual[maior_distancia], nao_evento_cal[maior_distancia],
               evento_cal[maior_distancia], color='gray')
axes[1].set_title(f'Estatística KS: {evento_cal[maior_distancia] - nao_evento_cal[maior_distancia]:.3f}')
axes[1].set_xlabel('% da base, da maior para a menor probabilidade')
axes[1].set_ylabel('% acumulado')
axes[1].legend()

plt.tight_layout()
plt.show()

*Captura de reinternações por faixa de prioridade (decis de probabilidade)*

In [ ]:
# Como a equipe só acompanha parte das altas, a pergunta prática é quantas reinternações ficam dentro dos X% de maior risco
ordem = np.argsort(-p_hat_calibrada, kind='stable')
y_ordenado = y_binaria[ordem]
p_ordenada = p_hat_calibrada[ordem]
decil = np.arange(len(y_ordenado)) * 10 // len(y_ordenado) + 1

tabela_decis = pd.DataFrame({'decil': decil, 'y': y_ordenado, 'p': p_ordenada}).groupby('decil').agg(
    qtd=('y', 'size'), reinternacoes=('y', 'sum'), taxa_observada=('y', 'mean'), prob_media=('p', 'mean'))
tabela_decis['perc_base_acumulado'] = tabela_decis['qtd'].cumsum() / tabela_decis['qtd'].sum()
tabela_decis['captura_acumulada'] = tabela_decis['reinternacoes'].cumsum() / tabela_decis['reinternacoes'].sum()
tabela_decis['lift'] = tabela_decis['taxa_observada'] / taxa_observada
tabela_decis

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(np.r_[0, tabela_decis['perc_base_acumulado']], np.r_[0, tabela_decis['captura_acumulada']],
         '-o', color='#C44E52', label='Modelo')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Seleção aleatória')
plt.xlabel('% das altas acompanhadas (da maior para a menor probabilidade)')
plt.ylabel('% das reinternações capturadas')
plt.title('Curva de captura no teste externo')
plt.legend()
plt.grid(linestyle='--', alpha=0.5)
plt.show()

*Função para as medidas de desempenho em um ponto de corte*

In [ ]:
def medidas_desempenho(y, p, corte):
    y_predito = (p >= corte).astype(int)
    vp = int(((y == 1) & (y_predito == 1)).sum())
    vn = int(((y == 0) & (y_predito == 0)).sum())
    fp = int(((y == 0) & (y_predito == 1)).sum())
    fn = int(((y == 1) & (y_predito == 0)).sum())
    sensibilidade = vp / (vp + fn) if vp + fn else 0
    especificidade = vn / (vn + fp) if vn + fp else 0
    ppv = vp / (vp + fp) if vp + fp else 0
    return {'corte': corte,
            'perc_sinalizados': (vp + fp) / len(y),
            'acuracia': (vp + vn) / len(y),
            'sensibilidade': sensibilidade,
            'especificidade': especificidade,
            'acuracia_balanceada': (sensibilidade + especificidade) / 2,
            'ppv': ppv,
            'npv': vn / (vn + fn) if vn + fn else 0,
            'f1': 2 * ppv * sensibilidade / (ppv + sensibilidade) if ppv + sensibilidade else 0}

*Medidas de desempenho em pontos de corte selecionados*

In [ ]:
y_teste_array = np.asarray(y_teste_externo)
taxa_geral = y_teste_array.mean()

# Corte que manda para o programa as 20% altas de maior risco; ajustar conforme a capacidade real da equipe
capacidade_acompanhamento = 0.20
corte_capacidade = np.quantile(p_hat_calibrada, 1 - capacidade_acompanhamento)

cortes = [0.50, 0.30, 0.20, 0.10, taxa_geral, corte_capacidade]
tabela_cortes = pd.DataFrame([medidas_desempenho(y_teste_array, p_hat_calibrada, k)
                              for k in cortes])
tabela_cortes

*Comparação das medidas de desempenho, antes e depois da calibração*

In [ ]:
comparacao_calibracao = pd.DataFrame({
    'Ponderadas, corte de 50%': medidas_desempenho(y_teste_array, p_hat, 0.50),
    'Calibradas, corte de 50%': medidas_desempenho(y_teste_array, p_hat_calibrada, 0.50),
    'Calibradas, corte da taxa geral': medidas_desempenho(y_teste_array, p_hat_calibrada, taxa_geral),
    'Calibradas, corte da capacidade': medidas_desempenho(y_teste_array, p_hat_calibrada, corte_capacidade)})
comparacao_calibracao

*Tabela de classificação no ponto de corte escolhido*

In [ ]:
corte_escolhido = corte_capacidade
y_predito = (p_hat_calibrada >= corte_escolhido).astype(int)
print(f"Corte escolhido: {corte_escolhido:.3f}")
print(pd.crosstab(y_teste_array, y_predito,
                  rownames=['Observado'], colnames=['Predito']))

### **Reajuste final do modelo**

*Reajuste do modelo na base completa (100% das observações)*

In [ ]:
pesos_completa = compute_sample_weight(class_weight='balanced', y=y)
modelo_final = clone(tecnicas[linha_modelo_escolhido["nome_tecnica"]][0]).set_params(**linha_modelo_escolhido["hiperparametros"])
modelo_final.fit(X_completa_tratada, y, sample_weight=pesos_completa)

*Calibração do modelo final, na base completa*

In [ ]:
modelo_calibrado_final = CalibratedClassifierCV(ModeloPonderado(modelo_final),
                                                method='isotonic',
                                                cv=5)
modelo_calibrado_final.fit(X_completa_tratada, y)

*Inspeção de perfis da árvore (aplicável apenas quando o modelo escolhido é de árvore de decisão)*

In [ ]:
def resumo_perfil(modelo, colunas, lista_quanti, lista_quali, preprocessador, categoria, posicao=1):
    """Resume as regras do nó terminal na posição solicitada do ranking de probabilidade predita
    da categoria informada: 1, 2, 3... = maiores probabilidades; -1, -2, -3... = menores."""
    tree = modelo.tree_
    indice_categoria = list(modelo.classes_).index(categoria)

    folhas = np.where(tree.children_left == -1)[0]
    valores = tree.value[folhas, 0, indice_categoria] / tree.value[folhas, 0, :].sum(axis=1)
    n_folhas = len(folhas)
    if posicao == 0 or abs(posicao) > n_folhas:
        print(f'Posição {posicao} inexistente: a árvore tem {n_folhas} nós terminais. '
              f'Use 1 a {n_folhas} (da maior probabilidade predita) ou -1 a -{n_folhas} (da menor).')
        return
    ranking = valores.argsort()[::-1]
    folha = folhas[ranking[posicao - 1]] if posicao > 0 else folhas[ranking[posicao]]

    pai = {}
    for no in range(tree.node_count):
        for filho, lado in ((tree.children_left[no], 'esq'), (tree.children_right[no], 'dir')):
            if filho != -1:
                pai[filho] = (no, lado)
    caminho, no = [], folha
    while no != 0:
        p, lado = pai[no]
        caminho.append((p, lado))
        no = p
    caminho.reverse()

    try:
        cats_por_var = dict(zip(lista_quali, preprocessador.named_transformers_['quali'].categories_))
    except (KeyError, AttributeError):
        cats_por_var = {}

    try:
        escalador = preprocessador.named_transformers_['quanti']
        escala_por_var = dict(zip(lista_quanti, zip(escalador.mean_, escalador.scale_)))
    except (KeyError, AttributeError):
        escala_por_var = {}
    def fmt(x):
        s = f'{x:.2f}'.rstrip('0').rstrip('.')
        return s.replace('.', ',')
    lim = {}
    quali_eq = {}
    quali_ne = {}
    ordem = []
    for no, lado in caminho:
        f, thr = colunas[tree.feature[no]], tree.threshold[no]
        if f in lista_quanti:
            var = f
            if var in escala_por_var:
                media, escala = escala_por_var[var]
                thr = thr * escala + media
            lo, hi = lim.get(var, [None, None])
            if lado == 'esq': hi = thr if hi is None else min(hi, thr)
            else:             lo = thr if lo is None else max(lo, thr)
            lim[var] = [lo, hi]
        else:
            var = max((v for v in lista_quali if f.startswith(v + '_')), key=len)
            cat = f[len(var) + 1:]
            if lado == 'dir':
                quali_eq[var] = cat
            else:
                outras = [c for c in cats_por_var.get(var, []) if c != cat]
                if len(outras) == 1: quali_eq.setdefault(var, outras[0])
                else:                quali_ne.setdefault(var, set()).add(cat)
        if var not in ordem: ordem.append(var)

    linhas = []
    for var in ordem:
        if var in lim:
            lo, hi = lim[var]
            if lo is not None and hi is not None: linhas.append(f'{var}: acima de {fmt(lo)} e até {fmt(hi)}')
            elif lo is not None:                  linhas.append(f'{var}: acima de {fmt(lo)}')
            else:                                 linhas.append(f'{var}: até {fmt(hi)}')
        elif var in quali_eq:
            linhas.append(f'{var}: "{quali_eq[var]}"')
        if var in quali_ne and var not in quali_eq:
            linhas.append(f'{var}: diferente de ' + ', '.join(f'"{c}"' for c in sorted(quali_ne[var])))

    valor = tree.value[folha, 0, indice_categoria] / tree.value[folha, 0, :].sum()
    n = tree.n_node_samples[folha]
    total = tree.n_node_samples[0]
    if   posicao ==  1: rotulo = 'mais elevada'
    elif posicao == -1: rotulo = 'mais baixa'
    elif posicao  >  1: rotulo = f'{posicao}ª mais elevada'
    else:               rotulo = f'{-posicao}ª mais baixa'
    print(f'Perfil de probabilidade predita {rotulo}\n')
    for l in linhas:
        print(f'  • {l}')
    print(f'\nProbabilidade predita da categoria "{categoria}": {fmt(100 * valor)}%')
    print(f'Quantidade de observações: {n}')
    print(f'Percentual de observações: {100 * n / total:.2f}%'.replace('.', ','))

In [ ]:
if isinstance(modelo_final, DecisionTreeClassifier):
    resumo_perfil(modelo_final,
                  X_completa_tratada.columns,
                  lista_X_quanti,
                  lista_X_quali,
                  preprocessador,
                  categoria=1,
                  posicao=1,
                 )

*Gráfico final da árvore (aplicável apenas quando o modelo escolhido é de árvore de decisão)*

In [ ]:
if isinstance(modelo_final, DecisionTreeClassifier):

    plt.figure(figsize=(30, 12))
    plot_tree(modelo_final,
              feature_names=X_completa_tratada.columns,
              class_names=modelo_final.classes_.astype(str),
              filled=True,
              proportion=True,
              fontsize=10,
              rounded=True,
              precision=2)
    plt.title('Árvore do modelo final (na base completa)')
    plt.show()

### **Interpretabilidade do modelo**

*Cálculo da importância de cada variável por redução de impureza*

In [ ]:
# O Hist. Gradient Boosting não fornece essa medida
if hasattr(modelo_final, 'feature_importances_'):
    importancias_impureza = pd.Series(modelo_final.feature_importances_,
                                      index=X_completa_tratada.columns).sort_values()
else:
    importancias_impureza = None
    print('O algoritmo do modelo final não fornece importâncias por redução de impureza.')

*Gráfico de importâncias por redução de impureza*

In [ ]:
if importancias_impureza is not None:
    plt.figure(figsize=(8, 8))
    importancias_impureza.plot.barh(title="Importância das Variáveis por Redução de Impureza",
                                    color="#084734")
    plt.xlabel("Redução Média de Impureza nas Quebras")
    plt.tight_layout()
    plt.show()

*Cálculo da importância de cada variável por permutação*

In [ ]:
importancias_permutacao = permutation_importance(modelo_escolhido,
                                                 X_teste_externo_tratada,
                                                 y_teste_externo,
                                                 n_repeats=30,
                                                 random_state=123,
                                                 scoring='roc_auc')
importancias_permutacao = pd.Series(importancias_permutacao.importances_mean,
                                    index=X_teste_externo_tratada.columns).sort_values()
importancias_permutacao.sort_values(ascending=False).head(10)

*Gráfico de importâncias por permutação*

In [ ]:
plt.figure(figsize=(8, 8))
importancias_permutacao.plot.barh(title="Importância das Variáveis por Permutação",
                                  color="#084734")
plt.xlabel("Queda Média na AUC")
plt.tight_layout()
plt.show()

*Cálculo da importância de cada variável por SHAP*

In [ ]:
# TreeExplainer é exato para árvores mas o tempo cresce com o número delas, daí a amostra; AdaBoost precisa do explicador genérico
if isinstance(modelo_final, AdaBoostClassifier):
    X_shap = shap.sample(X_completa_tratada, 500, random_state=123)
    explainer = shap.Explainer(modelo_final.predict_proba, X_shap)
else:
    X_shap = shap.sample(X_completa_tratada, 3000, random_state=123)
    explainer = shap.TreeExplainer(modelo_final)
valores_shap = explainer(X_shap)

# Árvore, floresta e AdaBoost devolvem uma coluna por categoria (em probabilidade); os boostings devolvem uma só, em logito
valores_shap_categoria = valores_shap[:, :, 1] if valores_shap.values.ndim == 3 else valores_shap

*Gráfico de valores absolutos médios de SHAP*

In [ ]:
shap.plots.bar(valores_shap_categoria, max_display=15)

*Gráfico de direção do impacto baseado em SHAP (beeswarm)*

In [ ]:
shap.summary_plot(valores_shap_categoria, X_shap, max_display=10)

*Explicação individual: paciente de maior risco da amostra*

In [ ]:
indice_maior_risco = int(np.argmax(modelo_calibrado_final.predict_proba(X_shap)[:, 1]))
shap.plots.waterfall(valores_shap_categoria[indice_maior_risco], max_display=10)

### **Utilização do modelo em nova base de dados**

*Criação de novo objeto 'X'*

In [ ]:
# Mesmas colunas da base original; preparar_base aplica os mesmos tratamentos e variáveis criadas
X_novo = pd.DataFrame({
    'idade': [34, 81, 67, 72, 55],
    'sexo': ['F', 'M', 'F', 'M', 'F'],
    'categoria_plano': ['executivo', 'basico', 'intermediario', 'basico', 'intermediario'],
    'mora_sozinho': ['nao', 'sim', np.nan, 'sim', 'nao'],
    'distancia_hospital_km': [8.2, 42.0, 15.5, 96.3, 5.1],
    'indice_charlson': [0, 6, 3, 8, 1],
    'qtd_internacoes_12m': [0, 3, 1, 5, 0],
    'qtd_atendimentos_ps_6m': [0, 4, 2, 6, 1],
    'tipo_admissao': ['eletiva', 'urgencia', 'urgencia', 'urgencia', 'eletiva'],
    'clinica_responsavel': ['ortopedia', 'cardiologia', 'clinica_medica', 'oncologia', 'cirurgia_geral'],
    'tempo_internacao_dias': [1, 12, 6, 18, 2],
    'passou_uti': ['nao', 'sim', 'nao', 'sim', 'nao'],
    'realizou_cirurgia': ['sim', 'nao', 'nao', 'nao', 'sim'],
    'hemoglobina_alta': [np.nan, 9.8, 11.9, 8.7, 13.6],
    'sodio_alta': [np.nan, 131.0, 137.0, 129.0, 140.0],
    'dia_semana_alta': ['terca', 'sabado', 'quarta', 'domingo', 'quinta'],
    'destino_alta': ['domicilio', 'domicilio', 'home_care', 'domicilio', 'domicilio'],
    'qtd_medicamentos_alta': [2, 14, 8, 17, 3],
    'retorno_agendado': ['sim', 'nao', 'sim', 'nao', 'sim'],
})
X_novo = preparar_base(X_novo)[lista_X_quanti + lista_X_quali]

*Pré-processamento*

In [ ]:
X_novo_tratada = preprocessador.transform(X_novo)

*Aplicação do modelo final*

In [ ]:
probabilidades = modelo_calibrado_final.predict_proba(X_novo_tratada)[:, 1]
pd.DataFrame({'Probabilidade predita': probabilidades.round(4),
              'Encaminhar ao programa': np.where(probabilidades >= corte_escolhido, 'sim', 'nao')})

### **Resumo dos resultados**

In [ ]:
linha_20 = tabela_decis.loc[2]
resumo = pd.Series({
    'Modelo escolhido': f"Modelo {num_modelo_escolhido} ({linha_modelo_escolhido['nome_tecnica']})",
    'AUC teste interno (média CV)': round(linha_modelo_escolhido['roc_auc_media_teste_interno'], 3),
    'AUC teste externo (calibrado)': round(roc_auc_score(y_binaria, p_hat_calibrada), 3),
    'KS teste externo (calibrado)': round(np.max(evento_cal - nao_evento_cal), 3),
    'Brier antes / depois da calibração': f"{brier:.3f} / {brier_calibrado:.3f}",
    'Taxa de reinternação no teste': f"{taxa_geral:.1%}",
    'Reinternações capturadas nos 20% de maior risco': f"{linha_20['captura_acumulada']:.1%}",
    'Taxa observada nos 20% de maior risco': f"{tabela_decis.loc[1:2, 'reinternacoes'].sum() / tabela_decis.loc[1:2, 'qtd'].sum():.1%}",
    'Precisão / sensibilidade / F1 no corte escolhido': "{:.3f} / {:.3f} / {:.3f}".format(
        *[medidas_desempenho(y_teste_array, p_hat_calibrada, corte_escolhido)[k] for k in ('ppv', 'sensibilidade', 'f1')]),
    'Variáveis mais importantes (permutação)': ', '.join(importancias_permutacao.sort_values(ascending=False).head(5).index)
})
resumo.to_frame('Valor')

### **Conclusões de negócio**

**1. Dá para prever reinternação na alta?** Sim, com poder de ordenação moderado: o modelo escolhido ([preencher após execução: técnica]) chega a AUC de [preencher após execução: AUC teste externo] e KS de [preencher após execução: KS] no teste externo, com desempenho parecido entre treino, validação cruzada e teste externo (o filtro de 10% descartou os modelos instáveis).

**2. Como usar no programa de transição de cuidado.** O valor do modelo está em ordenar as altas pelo risco. Acompanhando os 20% de maior probabilidade, a equipe alcança [preencher após execução: % capturado nos 20%] das reinternações, contra 20% numa escolha aleatória, e a taxa de reinternação nesse grupo é [preencher após execução: taxa no top 20%] contra [preencher após execução: taxa geral] na base. O ponto de corte deve sair da capacidade da equipe (parâmetro `capacidade_acompanhamento`), não do 0,5: com evento raro, quase ninguém passa de 50% depois da calibração.

**3. Quem tem mais risco.** As variáveis que mais pesam são [preencher após execução: top 5 da permutação/SHAP]. Na análise bivariada, a taxa sobe com mais internações e idas ao PS no histórico, mais comorbidades, mais medicamentos, oncologia, admissão de urgência, passagem por UTI, alta para home care ou instituição de longa permanência, quem mora sozinho, alta sem retorno agendado e alta no fim de semana. Um cuidado: pacientes que moram longe aparecem com menos reinternação, o que provavelmente reflete reinternações em outros hospitais, que a base não capta, e não um risco menor de fato.

**4. Probabilidades.** A ponderação de classes infla as probabilidades; depois da calibração isotônica a média predita volta para perto da taxa real e o Brier cai de [preencher após execução: Brier antes] para [preencher após execução: Brier depois]. Isso permite ler o resultado como risco real de cada paciente.

**5. Ações sugeridas.** Fatores modificáveis aparecem entre os relevantes: garantir consulta de retorno agendada antes da alta, reforçar a revisão de prescrição para quem sai com muitos medicamentos e cobrir as altas de sábado e domingo.

**Limitações.** Reinternações em outros hospitais não são captadas, então a taxa real é maior que a observada e parte dos "0" são falsos negativos de registro; a base cobre um ano só (jul/2025 a jun/2026), sem validação fora do tempo; não há variáveis de adesão, suporte familiar além de `mora_sozinho` ou condição social. Próximos passos: validar em período posterior, monitorar a calibração ao longo do tempo e medir o efeito do programa (quem é acompanhado deveria reinternar menos, o que altera os dados futuros).